In [25]:
# dependencies

from Bio import AlignIO
from Bio import SeqIO
import re
import pandas as pd

In [26]:
# Converting BioCatNet fasta file to dataframe that includes the sequence and the family name

def biocatnet_fasta_to_df(input_fasta):
    data = {'accession': [], 'SFID': [], 'GI': [], 'taxonID': [], 'sequence': []}
    with open(input_fasta) as fp:
        for record in SeqIO.parse(fp, 'fasta'):
            accession = record.id
            sfid = re.findall(r"sfid\|(\d+)", record.id)
            gi = re.findall(r"gi\|(\d+)", record.id)
            taxonID = re.findall(r"taxonID\|(\d+)", record.id)
            sequence = str(record.seq)

            #Attach to dict.
            data['accession'].append(accession)
            data['SFID'].append(", ".join(sfid))
            data['GI'].append(", ".join(gi))
            data['taxonID'].append(", ".join(taxonID))
            data['sequence'].append(sequence)
    df = pd.DataFrame(data)
    return df

In [27]:
# Converting HMM search output for BioCatNet data (e = 0.00001) to FASTA file
input = '/home/alecia/chapter3-repo/output/hmm_search_1/biocatnet/4/ihelixhits_e_0p0001_z__T_.aln'
outfile_fasta = '/home/alecia/chapter3-repo/output/hmm_search_1/biocatnet/4/ihelixhits_e_0p0001_z__T_.fasta'

with open(input) as infile:
    alignment = AlignIO.read(infile, 'stockholm')

SeqIO.write(alignment, outfile_fasta, 'fasta')

23076

In [28]:
# Convert FASTA file into df of the results

df_results = biocatnet_fasta_to_df('/home/alecia/chapter3-repo/output/hmm_search_1/biocatnet/4/ihelixhits_e_0p0001_z__T_.fasta')
df_results

,accession,SFID,GI,taxonID,sequence
0,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,-EDIMAQCFVFFFAGFETSST--TMTFALLELAQH
1,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,-SDIIAQCFVFFIAGFETSSS--TMTFTMLELAQH
2,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,-SDIIAQCFVFFIAGFETSSS--TMTFTMLELAQH
3,sid|107075|pid|82008|hfid|211|sfid|6|gb|ERL945...,6,546684943,77166,-NDIIGQCFVFFIAGFETSSS--TMTFTMLELAQN
4,sid|85794|pid|63405|hfid|120|sfid|4|gi|3544699...,4,"354469982, 537224038",10029,DKDLRAEVDTFMFEGHDTTAS--GISWIFYALATH
...,...,...,...,...,...
23071,sid|12889|pid|9332|hfid|141|sfid|4|gb|EFN68668...,4,307180804,104421,--DIREEVDTFMFEGHDTTAM--GIMFALLLLAEH
23072,sid|12561|pid|9080|hfid|372|sfid|49|gi|5577501...,49,557750175,7370,------LALDLFLVGVDTTSV--AVSSTIYQLAKN
23073,sid|56070|pid|39649|hfid|398|sfid|52|gi|596683...,52,596683427,1213859,---------TFMFAGHDTTSI--LLQWAIYELSR-
23074,sid|54897|pid|38848|hfid|782|sfid|105|gi|66566...,105,665669230,1950,-EELIGVGGVLLAAGLDTTAN--MLGLGTFALLSN


In [29]:
# Open database and put into dataframe

df_database = biocatnet_fasta_to_df('/home/alecia/chapter3-repo/input/databases/biocatnet_p450.fasta')
df_database

,accession,SFID,GI,taxonID,sequence
0,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,2,557295506,38654,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...
1,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,2,602643091,176946,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...
2,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,2,602671004,176946,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...
3,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,2,565323585,8665,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...
4,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,2,602670425,176946,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...
...,...,...,...,...,...
52669,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,5006,"118351107, 164519821",5911,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...
52670,sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,5009,"164519833, 118380288",5911,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...
52671,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,5060,667644755,655819,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...
52672,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,5280,667652271,655819,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...


In [30]:
# Merge results with database to see which sequences are missing from results
## Accession cannot be used to merge on due to the presence of 'sub seq' 

merged_df = (pd.merge(df_database, df_results, how='left', on=['GI', 'SFID', 'taxonID'])
             .fillna(0))
merged_df

,accession_x,SFID,GI,taxonID,sequence_x,accession_y,sequence_y
0,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,2,557295506,38654,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...,0,0
1,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,2,602643091,176946,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...,0,0
2,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,2,602671004,176946,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...,0,0
3,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,2,565323585,8665,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...,0,0
4,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,2,602670425,176946,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...,0,0
...,...,...,...,...,...,...,...
52722,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,5006,"118351107, 164519821",5911,MSIIQLFIGFIIGTIIYQTVIKTLYLYYVYKRRYGKDIIVLFYPVI...,sid|14140|pid|10364|hfid|1674|sfid|5006|gi|118...,-KEIVNQYITVIFAGTDTTSH--LIGNILFELSRN
52723,sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,5009,"164519833, 118380288",5911,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,0,0
52724,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,5060,667644755,655819,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,0,0
52725,sid|85661|pid|63286|hfid|1696|sfid|5280|gi|667...,5280,667652271,655819,MFMLWIAAAVFGLIIIGRPLIVYLQDVKGFRKYPVQNFLSGISPLA...,0,0


In [31]:
# Create new dataframe from where there are no hits

no_hits = merged_df.loc[merged_df["sequence_y"] == 0, ["GI", "accession_x", "sequence_x", "SFID", "taxonID"]]
no_hits

,GI,accession_x,sequence_x,SFID,taxonID
0,557295506,sid|90155|pid|67114|hfid|29|sfid|2|gi|55729550...,MDAGTTGILILLLLIFVLCYLVLEINRKRAQLPAGPAPWPVLGNLW...,2,38654
1,602643091,sid|90154|pid|67113|hfid|29|sfid|2|gi|60264309...,GPTPWPFLGNLLQRDVLPVDRLYKKLTAKYGPIFTVWIGSKPMVAL...,2,176946
2,602671004,sid|90156|pid|67115|hfid|29|sfid|2|gi|60267100...,MELTWAGALLLLCIFFTLFSSFQIYKKKGQLPPGPTPWPFLGNLLQ...,2,176946
3,565323585,sid|90236|pid|67189|hfid|29|sfid|2|gb|ETE74148...,MELTWTGVLLLLCVLFILFSSFQMYKKKGQLPPGPTPWPILGNLLQ...,2,8665
4,602670425,sid|101660|pid|77115|hfid|29|sfid|2|gi|6026704...,MELTWAGALLLLSILITLFSSFQMYKKKGQLPPGPTPWPFLGNLLQ...,2,176946
...,...,...,...,...,...
52720,546303216,sid|99038|pid|74742|hfid|1668|sfid|809|gi|5463...,MHVAFVFAPRLLNHSVGAQQHRRRPVCAASPPPQTEAPLPTPVAIP...,809,2769
52721,"164519785, 146163051",sid|14117|pid|10345|hfid|1672|sfid|5003|gb|ABY...,MILLIIGLLIFSLFSYFAYLIFVKPYVRNKWYVKQYGKICKPIYYP...,5003,5911
52723,"164519833, 118380288",sid|14148|pid|10372|hfid|1677|sfid|5009|gb|ABY...,MITTILIALTFIGIALLLFKAFIQPLYRISFYTKQGLKQKFFVPFL...,5009,5911
52724,667644755,sid|14504|pid|10663|hfid|1684|sfid|5060|gi|667...,MSSHKPPEAVRRETSLQGQSMTESHCMLALSERLHDPLPGCVTASI...,5060,655819


In [32]:
# Family counts in results and no hits

families_hits = (df_results.value_counts(df_results['SFID'])
                 .to_csv('/home/alecia/chapter3-repo/analysis/round1_missing_biocatnet/round1_biocatnet_hits_families_0p0001.txt')
)

families_nohits = (no_hits.value_counts(no_hits['SFID'])
                   .to_csv('/home/alecia/chapter3-repo/analysis/round1_missing_biocatnet/round1_biocatnet_nohits_families_0p0001.txt')
)